# Example 1.2.2 / 2: Defining a Parameter with =

Companion notebook for **1.2.2 Computed Parameters** in the AMPL lab Documentation, section "Defining a Parameter with `=`". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Defining a Parameter with `=`

When `share` or `tot_dem` changes, AMPL recomputes everything downstream of it. `value()` reads
the scalar `tot_sh`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set DEST;                              # destinations
param share {DEST} >= 0, <= 100;       # percent of total demand
param tot_sh = sum {j in DEST} share[j];   # sum of the shares, ideally 100
param tot_dem >= 0;                    # total demand, all destinations
param demand {j in DEST} = share[j] * tot_dem / tot_sh;   # demand, corrected for tot_sh

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["DEST"] = ["FRA", "DET", "LAN"]
ampl.param["share"] = {"FRA": 50, "DET": 30, "LAN": 21}
ampl.param["tot_dem"] = 3000

In [5]:
# run.py
print(ampl.param["tot_sh"].value())
print(ampl.param["demand"].to_pandas())

101
          demand
DET   891.089109
FRA  1485.148515
LAN   623.762376


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
set DEST := FRA DET LAN;
param share := FRA 50  DET 30  LAN 21;
param tot_dem := 3000;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display tot_sh, demand;

tot_sh = 101

demand [*] :=
DET   891.089
FRA  1485.15
LAN   623.762
;

